# AI Career Launchpad — Run on Google Colab

Runs the entire project on Colab with a public URL via Cloudflare tunnel.

### You need 2 free API keys:
| Key | Where | Time |
|-----|-------|------|
| NVIDIA NIM | [build.nvidia.com](https://build.nvidia.com) → Pick model → Get API Key | 1 min |
| Adzuna | [developer.adzuna.com](https://developer.adzuna.com) → Sign up | 1 min |

### Run cells 1-7 in order. Total: ~4 minutes.

---
## 1. Enter API keys

In [ ]:
import getpass, os

print("=" * 50)
print("  API Key Setup")
print("=" * 50)
print()
print("Get your free keys from:")
print("  NVIDIA NIM  → https://build.nvidia.com")
print("  Adzuna      → https://developer.adzuna.com")
print()

NVIDIA_API_KEY = getpass.getpass("1. NVIDIA NIM API Key (starts with nvapi-): ")
ADZUNA_APP_ID = input("2. Adzuna App ID: ").strip()
ADZUNA_APP_KEY = getpass.getpass("3. Adzuna App Key: ")

errors = []
if not NVIDIA_API_KEY.startswith("nvapi-"):
    errors.append("  NVIDIA key should start with 'nvapi-'")
if not ADZUNA_APP_ID:
    errors.append("  Adzuna App ID is empty")
if not ADZUNA_APP_KEY:
    errors.append("  Adzuna App Key is empty")

if errors:
    print()
    print("Warnings:")
    for e in errors:
        print(e)
    print("Saved anyway — fix and re-run this cell if needed.")

os.makedirs("/content/career-launchpad", exist_ok=True)
with open("/content/career-launchpad/.env", "w") as f:
    f.write(f"""NVIDIA_API_KEY={NVIDIA_API_KEY}
NVIDIA_BASE_URL=https://integrate.api.nvidia.com/v1
NVIDIA_MODEL=meta/llama-3.3-70b-instruct
ADZUNA_APP_ID={ADZUNA_APP_ID}
ADZUNA_APP_KEY={ADZUNA_APP_KEY}
PORT=3001
RAG_SERVICE_URL=http://localhost:8000
NODE_ENV=development
""")

print()
print(f"  NVIDIA key: nvapi-...{NVIDIA_API_KEY[-4:]}")
print(f"  Adzuna ID:  {ADZUNA_APP_ID}")
print(f"  Adzuna key: ...{ADZUNA_APP_KEY[-4:]}")
print()
print("Keys saved. Run next cell.")

---
## 2. Install Node.js + Python packages

In [ ]:
%%bash
echo "Installing Node.js 18..."
curl -fsSL https://deb.nodesource.com/setup_18.x | sudo -E bash - > /dev/null 2>&1
sudo apt-get install -y nodejs > /dev/null 2>&1
echo "Node $(node --version) / npm $(npm --version)"
echo ""
echo "Installing Python packages..."
pip install -q fastapi uvicorn chromadb sentence-transformers openai python-dotenv pydantic requests pypdf python-multipart
echo ""
echo "Installing Cloudflare tunnel..."
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared
echo "Done."

---
## 3. Upload the project zip
Click upload and select `ai-career-launchpad.zip`

In [ ]:
from google.colab import files
import zipfile, shutil, os

print("Upload ai-career-launchpad.zip:")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

for d in ["client", "server", "rag-service"]:
    p = f"/content/career-launchpad/{d}"
    if os.path.exists(p): shutil.rmtree(p)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

env_path = "/content/career-launchpad/.env"
if os.path.exists(env_path):
    shutil.copy(env_path, "/content/career-launchpad/server/.env")
    shutil.copy(env_path, "/content/career-launchpad/rag-service/.env")

print("Project extracted:")
!ls /content/career-launchpad/

---
## 4. Install npm packages + seed database

In [ ]:
%%bash
echo "=== Server npm install ==="
cd /content/career-launchpad/server && npm install --silent 2>&1 | tail -2
echo ""
echo "=== Client npm install ==="
cd /content/career-launchpad/client && npm install --silent 2>&1 | tail -2
echo ""
echo "=== Seeding database (downloads embedding model ~80MB first time) ==="
cd /content/career-launchpad/rag-service && python seed_data.py
echo ""
echo "All ready."

---
## 5. Build React frontend

In [ ]:
%%bash
cd /content/career-launchpad/client && npx vite build 2>&1 | tail -5
echo "Frontend built."

---
## 6. Start all services

In [ ]:
import subprocess, time, os

for port in [8000, 3001, 5000]:
    os.system(f"fuser -k {port}/tcp 2>/dev/null")
time.sleep(1)

env = {**os.environ}
with open("/content/career-launchpad/.env") as f:
    for line in f:
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            k, v = line.split("=", 1)
            env[k] = v

print("1/3 Starting RAG service (port 8000)...")
subprocess.Popen(
    ["python", "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/career-launchpad/rag-service",
    env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(8)
import requests
try:
    r = requests.get("http://localhost:8000/health", timeout=5)
    print(f"     {r.json()}")
except:
    print("     Still loading... wait 30s then run next cell.")

print("2/3 Starting Express backend (port 3001)...")
subprocess.Popen(
    ["node", "index.js"],
    cwd="/content/career-launchpad/server",
    env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(2)
print("     Running.")

SERVE = 'import http.server,socketserver,urllib.request,json,os\nPORT=5000\nSTATIC="/content/career-launchpad/client/dist"\nAPI="http://localhost:3001"\nclass H(http.server.SimpleHTTPRequestHandler):\n def __init__(s,*a,**k):super().__init__(*a,directory=STATIC,**k)\n def do_GET(s):\n  if s.path.startswith("/api/"):return s._proxy()\n  fp=os.path.join(STATIC,s.path.lstrip("/"))\n  if os.path.isfile(fp):super().do_GET()\n  else:s.path="/index.html";super().do_GET()\n def do_POST(s):\n  if s.path.startswith("/api/"):return s._proxy()\n  s.send_error(404)\n def _proxy(s):\n  url=f"{API}{s.path}"\n  try:\n   cl=int(s.headers.get("Content-Length",0))\n   body=s.rfile.read(cl) if cl>0 else None\n   ct=s.headers.get("Content-Type","application/json")\n   req=urllib.request.Request(url,data=body,headers={"Content-Type":ct},method=s.command)\n   with urllib.request.urlopen(req,timeout=120) as resp:\n    data=resp.read()\n    s.send_response(resp.status)\n    s.send_header("Content-Type",resp.headers.get("Content-Type","application/json"))\n    s.send_header("Content-Length",len(data))\n    s.end_headers()\n    s.wfile.write(data)\n  except Exception as e:\n   err=json.dumps({"error":str(e)}).encode()\n   s.send_response(502)\n   s.send_header("Content-Type","application/json")\n   s.send_header("Content-Length",len(err))\n   s.end_headers()\n   s.wfile.write(err)\n def log_message(s,*a):pass\nwith socketserver.TCPServer(("",PORT),H) as sv:\n print(f"Serving on {PORT}")\n sv.serve_forever()'
with open("/content/_serve.py", "w") as f:
    f.write(SERVE.replace('\\n','\n'))

print("3/3 Starting combined server (port 5000)...")
subprocess.Popen(["python", "/content/_serve.py"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(2)
print("     Running.")
print()
print("All services up. Run next cell for public URL.")

---
## 7. Get your public URL (Cloudflare tunnel)
**Click the URL to open your app!** No account needed.

In [ ]:
import subprocess, time, re

# Kill any old tunnel
import os
os.system("pkill -f cloudflared 2>/dev/null")
time.sleep(2)

# Start Cloudflare tunnel
proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:5000"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

# Read URL from output
url = None
for _ in range(30):
    line = proc.stderr.readline()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            break
    time.sleep(0.5)

if url:
    print("=" * 55)
    print()
    print("   YOUR APP IS LIVE!")
    print()
    print(f"   {url}")
    print()
    print("=" * 55)
    print()
    print("Open that URL in your browser.")
    print("No passwords, no sign-up needed.")
    print("Keep this Colab tab open while using the app.")
else:
    print("Tunnel still starting... wait 10 seconds and re-run this cell.")

---
## Optional: Test backend

In [ ]:
import requests

for name, url in [("RAG", "http://localhost:8000/health"), ("Express", "http://localhost:3001/api/health"), ("Proxy", "http://localhost:5000/api/health")]:
    try:
        print(f"{name}: {requests.get(url, timeout=5).json()}")
    except Exception as e:
        print(f"{name}: DOWN — {e}")

---
## Stop everything

In [ ]:
import os
os.system("pkill -f cloudflared 2>/dev/null")
for p in [8000, 3001, 5000]:
    os.system(f"fuser -k {p}/tcp 2>/dev/null")
print("All stopped.")
print("Now go to: Runtime > Disconnect and delete runtime")
